In [4]:
def naca2412(n=80):

    x = np.linspace(0,1,n)

    t = 0.12
    m = 0.02
    p = 0.4

    yt = 5*t*(
        0.2969*np.sqrt(x)
        -0.1260*x
        -0.3516*x**2
        +0.2843*x**3
        -0.1015*x**4
    )

    yc = np.where(
        x < p,
        m/p**2 * (2*p*x - x**2),
        m/(1-p)**2 * ((1-2*p)+2*p*x-x**2)
    )

    dyc_dx = np.where(
        x < p,
        2*m/p**2 * (p-x),
        2*m/(1-p)**2 * (p-x)
    )

    theta = np.arctan(dyc_dx)

    xu = x - yt*np.sin(theta)
    yu = yc + yt*np.cos(theta)

    xl = x + yt*np.sin(theta)
    yl = yc - yt*np.cos(theta)

    # CLOCKWISE ordering
    x_airfoil = np.concatenate([xu[::-1], xl[1:]])
    y_airfoil = np.concatenate([yu[::-1], yl[1:]])

    return np.column_stack((x_airfoil, y_airfoil))

In [5]:
import numpy as np
import numpy.linalg as la
import matplotlib.pyplot as plt


def vortex_panel(coords, alpha_deg):

    # -------------------------
    # geometry
    # -------------------------
    x = coords[:,0]
    y = coords[:,1]

    num_panels = len(x) - 1

    alpha = np.radians(alpha_deg)

    # control points
    x_c = 0.5*(x[:-1] + x[1:])
    y_c = 0.5*(y[:-1] + y[1:])

    # panel geometry
    dx = np.diff(x)
    dy = np.diff(y)

    S = np.sqrt(dx**2 + dy**2)
    theta = np.arctan2(dy, dx)

    # -------------------------
    # influence matrices
    # -------------------------
    C_n1 = np.zeros((num_panels, num_panels))
    C_n2 = np.zeros((num_panels, num_panels))

    C_t1 = np.zeros((num_panels, num_panels))
    C_t2 = np.zeros((num_panels, num_panels))

    A_n = np.zeros((num_panels+1, num_panels+1))
    A_t = np.zeros((num_panels,   num_panels+1))

    RHS = np.zeros(num_panels+1)

    # -------------------------
    # build influence coeffs
    # -------------------------
    for i in range(num_panels):

        for j in range(num_panels):

            if i == j:

                # self influence
                C_n1[i,j] = -1.0
                C_n2[i,j] =  1.0

                C_t1[i,j] = 0.5*np.pi
                C_t2[i,j] = 0.5*np.pi

            else:

                A = -(x_c[i]-x[j])*np.cos(theta[j]) \
                    -(y_c[i]-y[j])*np.sin(theta[j])

                B = (x_c[i]-x[j])**2 \
                    +(y_c[i]-y[j])**2

                C = np.sin(theta[i]-theta[j])

                D = np.cos(theta[i]-theta[j])

                E = (x_c[i]-x[j])*np.sin(theta[j]) \
                    -(y_c[i]-y[j])*np.cos(theta[j])

                F = np.log(1 + S[j]*(S[j]+2*A)/B)

                G = np.arctan2(E*S[j], B + A*S[j])

                P = (x_c[i]-x[j]) * np.sin(theta[i]-2*theta[j]) \
                    + (y_c[i]-y[j]) * np.cos(theta[i]-2*theta[j])

                Q = (x_c[i]-x[j]) * np.cos(theta[i]-2*theta[j]) \
                    - (y_c[i]-y[j]) * np.sin(theta[i]-2*theta[j])

                # normal influence
                C_n2[i,j] = D \
                    + 0.5*Q*F/S[j] \
                    - (A*C + D*E)*G/S[j]

                C_n1[i,j] = 0.5*D*F \
                    + C*G \
                    - C_n2[i,j]

                # tangential influence
                C_t2[i,j] = C \
                    + 0.5*P*F/S[j] \
                    + (A*D - C*E)*G/S[j]

                C_t1[i,j] = 0.5*C*F \
                    - D*G \
                    - C_t2[i,j]

        # -------------------------
        # assemble A_n and A_t
        # -------------------------
        A_n[i,0] = C_n1[i,0]
        A_t[i,0] = C_t1[i,0]

        for j in range(1, num_panels):

            A_n[i,j] = C_n1[i,j] + C_n2[i,j-1]
            A_t[i,j] = C_t1[i,j] + C_t2[i,j-1]

        A_n[i,num_panels] = C_n2[i,num_panels-1]
        A_t[i,num_panels] = C_t2[i,num_panels-1]

        RHS[i] = np.sin(theta[i] - alpha)

    # -------------------------
    # Kutta condition
    # -------------------------
    A_n[num_panels,0] = 1.0
    A_n[num_panels,num_panels] = 1.0

    RHS[num_panels] = 0.0

    # -------------------------
    # solve system
    # -------------------------
    Gamma = la.solve(A_n, RHS)

    # -------------------------
    # tangential velocity
    # -------------------------
    V = np.zeros(num_panels)

    for i in range(num_panels):

        V[i] = np.cos(theta[i] - alpha) \
             + np.dot(A_t[i,:], Gamma)

    # -------------------------
    # pressure coefficient
    # -------------------------
    Cp = 1 - V**2

    # -------------------------
    # circulation + lift
    # -------------------------
    gamma_panel = 0.5*(Gamma[:-1] + Gamma[1:])

    Gamma_total = np.sum(gamma_panel * S)

    chord = np.max(x) - np.min(x)

    # because gamma' = gamma/(2*pi*Vinf)
    CL = 4*np.pi * Gamma_total / chord

    # -------------------------
    # diagnostics
    # -------------------------
    print('CL =', CL)
    print('V min/max:', V.min(), V.max())
    print('Cp min/max:', Cp.min(), Cp.max())

    # -------------------------
    # split upper/lower surface
    # -------------------------
    mid = num_panels // 2

    x_lower = x_c[:mid]
    Cp_lower = Cp[:mid]

    x_upper = x_c[mid:]
    Cp_upper = Cp[mid:]

    # sort for plotting
    idx_l = np.argsort(x_lower)
    idx_u = np.argsort(x_upper)

    # -------------------------
    # Cp plot
    # -------------------------
    plt.figure(figsize=(9,5))

    plt.plot(
        x_upper[idx_u],
        Cp_upper[idx_u],
        'bo-',
        label='Upper Surface'
    )

    plt.plot(
        x_lower[idx_l],
        Cp_lower[idx_l],
        'ro-',
        label='Lower Surface'
    )

    plt.gca().invert_yaxis()

    plt.xlabel('x/c')
    plt.ylabel('$C_p$')

    plt.title(
        f'Cp Distribution - α={alpha_deg}°, CL={CL:.3f}'
    )

    plt.grid(True)
    plt.legend()

    plt.xlim(0,1)
    plt.ylim(1.5,-3)

    plt.show()

    return Cp, CL, x_c, V, Gamma

In [6]:
import numpy as np
import numpy.linalg as la
import matplotlib.pyplot as plt

# IMPORTANT FOR JUPYTER
%matplotlib inline


def vortex_panel(coords, alpha_deg):

    x = coords[:,0]
    y = coords[:,1]

    num_panels = len(x) - 1

    alpha = np.radians(alpha_deg)

    # ---------------------------------
    # control points
    # ---------------------------------
    x_c = 0.5*(x[:-1] + x[1:])
    y_c = 0.5*(y[:-1] + y[1:])

    # ---------------------------------
    # panel geometry
    # ---------------------------------
    dx = np.diff(x)
    dy = np.diff(y)

    S = np.sqrt(dx**2 + dy**2)

    theta = np.arctan2(dy, dx)

    # ---------------------------------
    # matrices
    # ---------------------------------
    C_n1 = np.zeros((num_panels, num_panels))
    C_n2 = np.zeros((num_panels, num_panels))

    C_t1 = np.zeros((num_panels, num_panels))
    C_t2 = np.zeros((num_panels, num_panels))

    A_n = np.zeros((num_panels+1, num_panels+1))
    A_t = np.zeros((num_panels, num_panels+1))

    RHS = np.zeros(num_panels+1)

    # ---------------------------------
    # influence coefficients
    # ---------------------------------
    for i in range(num_panels):

        for j in range(num_panels):

            if i == j:

                C_n1[i,j] = -1.0
                C_n2[i,j] = 1.0

                C_t1[i,j] = 0.5*np.pi
                C_t2[i,j] = 0.5*np.pi

            else:

                A = -(x_c[i]-x[j])*np.cos(theta[j]) \
                    -(y_c[i]-y[j])*np.sin(theta[j])

                B = (x_c[i]-x[j])**2 \
                    +(y_c[i]-y[j])**2

                C = np.sin(theta[i]-theta[j])

                D = np.cos(theta[i]-theta[j])

                E = (x_c[i]-x[j])*np.sin(theta[j]) \
                    -(y_c[i]-y[j])*np.cos(theta[j])

                F = np.log(
                    1 + S[j]*(S[j]+2*A)/B
                )

                G = np.arctan2(
                    E*S[j],
                    B + A*S[j]
                )

                P = (x_c[i]-x[j]) * np.sin(theta[i]-2*theta[j]) \
                    + (y_c[i]-y[j]) * np.cos(theta[i]-2*theta[j])

                Q = (x_c[i]-x[j]) * np.cos(theta[i]-2*theta[j]) \
                    - (y_c[i]-y[j]) * np.sin(theta[i]-2*theta[j])

                # normal influence
                C_n2[i,j] = D \
                    + 0.5*Q*F/S[j] \
                    - (A*C + D*E)*G/S[j]

                C_n1[i,j] = 0.5*D*F \
                    + C*G \
                    - C_n2[i,j]

                # tangential influence
                C_t2[i,j] = C \
                    + 0.5*P*F/S[j] \
                    + (A*D - C*E)*G/S[j]

                C_t1[i,j] = 0.5*C*F \
                    - D*G \
                    - C_t2[i,j]

        # assemble system
        A_n[i,0] = C_n1[i,0]
        A_t[i,0] = C_t1[i,0]

        for j in range(1, num_panels):

            A_n[i,j] = C_n1[i,j] + C_n2[i,j-1]

            A_t[i,j] = C_t1[i,j] + C_t2[i,j-1]

        A_n[i,num_panels] = C_n2[i,num_panels-1]
        A_t[i,num_panels] = C_t2[i,num_panels-1]

        RHS[i] = np.sin(theta[i] - alpha)

    # ---------------------------------
    # Kutta condition
    # ---------------------------------
    A_n[num_panels,0] = 1.0
    A_n[num_panels,num_panels] = 1.0

    RHS[num_panels] = 0.0

    # ---------------------------------
    # solve gamma
    # ---------------------------------
    Gamma = la.solve(A_n, RHS)

    # ---------------------------------
    # tangential velocity
    # ---------------------------------
    V = np.zeros(num_panels)

    for i in range(num_panels):

        V[i] = np.cos(theta[i] - alpha) \
             + np.dot(A_t[i,:], Gamma)

    # ---------------------------------
    # pressure coefficient
    # ---------------------------------
    Cp = 1 - V**2

    # ---------------------------------
    # lift coefficient
    # ---------------------------------
    gamma_panel = 0.5*(Gamma[:-1] + Gamma[1:])

    Gamma_total = np.sum(gamma_panel * S)

    chord = np.max(x) - np.min(x)

    CL = 4*np.pi * Gamma_total / chord

    print("CL =", CL)

    print("V min/max =", V.min(), V.max())

    print("Cp min/max =", Cp.min(), Cp.max())

    # ---------------------------------
    # split surfaces
    # ---------------------------------
    mid = num_panels // 2

    x_lower = x_c[:mid]
    Cp_lower = Cp[:mid]

    x_upper = x_c[mid:]
    Cp_upper = Cp[mid:]

    idx_l = np.argsort(x_lower)
    idx_u = np.argsort(x_upper)

    # ---------------------------------
    # plot
    # ---------------------------------
    plt.figure(figsize=(9,5))

    plt.plot(
        x_upper[idx_u],
        Cp_upper[idx_u],
        'bo-',
        label='Upper Surface'
    )

    plt.plot(
        x_lower[idx_l],
        Cp_lower[idx_l],
        'ro-',
        label='Lower Surface'
    )

    plt.gca().invert_yaxis()

    plt.xlabel('x/c')
    plt.ylabel('$C_p$')

    plt.title(
        f'NACA Airfoil Cp Distribution\n'
        f'α = {alpha_deg}°, CL = {CL:.3f}'
    )

    plt.grid(True)
    plt.legend()

    plt.xlim(0,1)
    plt.ylim(1.5,-3)

    plt.show()

    return Cp, CL


# ==========================================
# RUN THE SOLVER
# ==========================================

Cp, CL = vortex_panel(coords, alpha_deg=0)

NameError: name 'coords' is not defined